# Data Loader Scratchpad

This notebook is a quick sanity check for the current EEG, MEG, and fMRI loaders.
It uses the repo APIs directly, so the displayed split sizes stay aligned with the codebase.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))


            import pandas as pd

            from src.eval_utils import create_dataset, load_config

            config = load_config(ROOT / "config.yaml")


In [ ]:
def summarize_dataset(modality, subject, split, shared_only=False, shared_manifest=None):
    config_local = load_config(ROOT / "config.yaml")
    if shared_manifest:
        config_local.setdefault("data", {})["shared_manifest_path"] = str(ROOT / shared_manifest)
    dataset = create_dataset(
        config_local,
        modality,
        split,
        subject=subject,
        shared_only=shared_only,
        quiet=True,
    )
    if modality == "eeg":
        unique_images = len({Path(path).stem for path in dataset.files})
        unique_concepts = len({concept for concept in dataset.concepts})
        trial_count = len(dataset)
    else:
        unique_images = len({trial["image_id"] for trial in dataset.trials})
        unique_concepts = len({trial["image_id"].rsplit("_", 1)[0] for trial in dataset.trials})
        trial_count = len(dataset.trials)
    sample = dataset[0]["x"]
    return {
        "modality": modality,
        "subject": subject,
        "split": split,
        "shared_only": shared_only,
        "unique_images": unique_images,
        "unique_concepts": unique_concepts,
        "trial_count": trial_count,
        "sample_shape": tuple(sample.shape),
    }

rows = []
for modality, subject in [("eeg", 1), ("meg", 1), ("fmri", 1)]:
    for split in ["train", "val", "test"]:
        rows.append(summarize_dataset(modality, subject, split))

pd.DataFrame(rows)
